In [1]:
# 加载所需要的包
%matplotlib inline

# scitnific computing and plotting
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

# HDDM related packages
import pymc as pm
import hddm
import kabuki
import arviz as az
print("The current HDDM version is: ", hddm.__version__)
print("The current kabuki version is: ", kabuki.__version__)
print("The current PyMC version is: ", pm.__version__)
print("The current ArviZ version is: ", az.__version__)

import warnings
warnings.filterwarnings('ignore') 

The current HDDM version is:  1.0.1RC
The current kabuki version is:  0.6.5RC4
The current PyMC version is:  2.3.8
The current ArviZ version is:  0.15.1


In [2]:
data = hddm.load_csv('../../../3_Data/exp3/CleanData/data_all.csv')
#剔除缺失值
# data_rdk = data_rdk.dropna(subset=['association'])

In [4]:
data

,subj_idx,rt,correct,part,response,association,difficulty,group,task_type
0,2,1.592,0,RDK_shape,f,self,hard,shape,relevant
1,2,1.821,1,RDK_shape,f,self,easy,shape,relevant
2,2,2.190,1,RDK_shape,f,other,easy,shape,relevant
3,2,2.177,1,RDK_shape,j,self,easy,shape,relevant
4,2,2.228,0,RDK_shape,f,other,hard,shape,relevant
...,...,...,...,...,...,...,...,...,...
712,4,2.522,1,RDK_shape,j,self,easy,shape,relevant
713,4,2.730,1,RDK_shape,f,self,hard,shape,relevant
714,4,3.944,1,RDK_shape,f,other,hard,shape,relevant
715,4,2.574,1,RDK_shape,f,other,easy,shape,relevant


In [5]:
# # 读取数据并进行预处理，response=choice
# # data_color = hddm.load_csv('../../3_Data/exp1a/CleanData/data_color.csv')

# data_match = data_match.assign(
#     correct=data_match['correct'].astype(int),
#     response=data_match['response'].map({'f': 1, 'j': 0}),
#     choice=data_match['response'].map({'f': 1, 'j': 0}),
#     stim = data_match['matchness'].map({'match':1, 'mismatch':-1})
# )

# data_rdk = data_rdk.assign(
#     correct=data_rdk['correct'].astype(int),
#     difficulty = data_rdk['difficulty'].map({'easy':0, 'hard':1}),
    
#     # 按键编码：d=1, k=0, 左箭头=1, 右箭头=0
#     response=data_rdk['response'].map({
#         'd': 1, 
#         'k': 0, 
#         'arrowleft': 1, 
#         'arrowright': 0
#     }),
    
#     # 运动方向：左180→1，右0→-1
#     # coherent_direction=data_rdk['coherent_direction'].map({180: 1, 0: -1}),
    
#     # 合并运动方向 + 颜色编码 stim：左和红为1，右和蓝为0
#    stim=data_rdk.apply(
#     lambda row:
#         # 运动任务：只按方向判断
#         1 if (row['part'] == 'RDK_motion' and row['coherent_direction'] == 180) else
#         0 if (row['part'] == 'RDK_motion' and row['coherent_direction'] == 0) else
#         # 颜色任务：只按颜色判断
#         1 if (row['part'] == 'RDK_color' and row['dot_color_final'] == '["hsl(0, 50%, 50%)","hsl(225, 50%, 50%)"]') else
#         0, # if (row['part'] == 'RDK_colorr' and row['dot_color_final'] == '["hsl(225, 50%, 50%)","hsl(0, 50%, 50%)"]') else # 这里有点问题，不这样写会得空值
#         # np.nan,  # 都不满足就标记缺失
#     axis=1
#    )
# )

# data_rdk

## 基础模型

In [6]:
# # 构建回归模型
# model_reg = [
#     {'model': "v ~ 1", 'link_func': lambda x: x}
# ] 
 
# m0 = hddm.HDDMRegressor(
#     data,
#     model_reg,
#     bias=True,
#     include=['v', 'a', 't', 'z'],
#     group_only_regressors=False
# )

# m0_idata = rdk_task_m1.sample(5000, burn=2000, chains=4, return_infdata=True, dbname = 'm0.db', db = 'pickle')
# m0.save('m0.hddm')

In [55]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(m1_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                                                  mean     sd  hdi_3%   
a                                                2.291  0.067   2.160  \
a_std                                            0.503  0.053   0.408   
t                                                0.465  0.016   0.436   
t_std                                            0.113  0.013   0.088   
z_Intercept                                      0.506  0.006   0.495   
z_Intercept_std                                  0.039  0.005   0.029   
v_Intercept                                      0.620  0.039   0.545   
v_Intercept_std                                  0.245  0.031   0.192   
v_association[T.self]                            0.041  0.033  -0.018   
v_association[T.self]_std                        0.104  0.043   0.020   
v_task_type[T.relevant]                          0.044  0.064  -0.076   
v_task_type[T.relevant]_std                      0.434  0.052   0.334   
v_association[T.self]:task_type[T.relevant]      0.

## m1：v随关联类型、任务类型和难度变化

In [7]:
# # 构建回归模型
# model_reg = [
#     {'model': "v ~ 1 + association*task_type*difficulty", 'link_func': lambda x: x}
# ] 
 
# m1 = hddm.HDDMRegressor(
#     data,
#     model_reg,
#     bias=True,
#     include=['v', 'a', 't', 'z'],
#     group_only_regressors=False
# )

# m1_idata = m1.sample(5000, burn=2000, chains=4, return_infdata=True, dbname = 'm1.db', db = 'pickle')
# m1.save('m1.hddm')

In [62]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(m1_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                                                     mean     sd  hdi_3%   
a                                                   2.310  0.067   2.183  \
a_std                                               0.510  0.053   0.417   
t                                                   0.463  0.015   0.435   
t_std                                               0.112  0.013   0.090   
z_Intercept                                         0.506  0.006   0.495   
z_Intercept_std                                     0.039  0.005   0.030   
v_Intercept                                         0.790  0.045   0.709   
v_Intercept_std                                     0.251  0.031   0.194   
v_association[T.self]                               0.076  0.050  -0.017   
v_association[T.self]_std                           0.097  0.051   0.008   
v_task_type[T.relevant]                             0.090  0.073  -0.051   
v_task_type[T.relevant]_std                         0.460  0.055   0.367   
v_associatio

### m2：在m1的基础上，将不同的组作为协变量纳入

In [8]:
# # 构建回归模型
# model_reg = [
#     {'model': "v ~ 1 + association*task_type*difficulty*group", 'link_func': lambda x: x}
# ] 
 
# m2 = hddm.HDDMRegressor(
#     data,
#     model_reg,
#     bias=True,
#     include=['v', 'a', 't', 'z'],
#     group_only_regressors=False
# )

# m2_idata = m2.sample(5000, burn=2000, chains=4, return_infdata=True, dbname = 'm2.db', db = 'pickle')
# m2.save('rdk_task_m3.hddm')

In [10]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(m2_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                                                     mean     sd  hdi_3%   
a                                                   2.320  0.064   2.201  \
a_std                                               0.487  0.050   0.397   
t                                                   0.454  0.016   0.423   
t_std                                               0.121  0.013   0.099   
z_Intercept                                         0.504  0.006   0.493   
z_Intercept_std                                     0.037  0.005   0.028   
v_Intercept                                         0.527  0.056   0.419   
v_Intercept_std                                     0.230  0.029   0.176   
v_association[T.self]                               0.112  0.048   0.027   
v_association[T.self]_std                           0.061  0.040   0.000   
v_task_type[T.relevant]                             0.446  0.082   0.296   
v_task_type[T.relevant]_std                         0.376  0.058   0.272   
v_group[T.mo

## 模型比较

In [ ]:
DIC_dict = {
    "m0": m0.dic,
    "m1": m1.dic,
    "m2": m2.dic
}

DIC_table = pd.DataFrame.from_dict(DIC_dict, orient="index", columns=["DIC"])
DIC_table["model"] = DIC_table.index
DIC_table = DIC_table[["model", "DIC"]]
DIC_table.sort_values(by=["DIC"], ascending=False)